## 先复现交互图
位置从 0 开始：2 代表“坐”。当前 token 只能看到自己和过去。分数是教学用假设值；不是模型输出。先猜三个未来位置的权重。

In [ ]:
import math
tokens = ["小", "猫", "坐", "在", "垫", "上"]
scores = [0, 1.2, 0.3, 0.7, 0.9, 0.2]
position = 2

def causal_weights(scores, position):
    visible = scores[:position + 1]
    largest = max(visible)
    exps = [math.exp(s - largest) for s in visible]
    denominator = sum(exps)
    return [x / denominator for x in exps] + [0.0] * (len(scores) - len(visible))

weights = causal_weights(scores, position)
for token, weight in zip(tokens, weights):
    print(token, f"{weight:.1%}")
print("权重和:", sum(weights))


## 为什么一个位置的权重会影响其他位置？
把“猫”的分数增加 2。其他分数没动，但总权重始终是 1。先预测“小”的权重，再运行。减去最大分数保持 softmax 结果不变，并降低指数溢出的风险。

In [ ]:
changed = scores.copy()
changed[1] += 2
new_weights = causal_weights(changed, position)
for token, before, after in zip(tokens, weights, new_weights):
    print(token, f"{before:.1%} -> {after:.1%}")


## 真正读出来的是 V 的加权和
让每个位置的 V 暂时只有一个分量，分别是 1 到 6。真实 Attention 中每个 V 都是向量，但每个分量仍做同样的乘法和求和。

In [ ]:
values = [1, 2, 3, 4, 5, 6]
context = sum(weight * value for weight, value in zip(weights, values))
print("读取结果:", context)
assert abs(sum(weights) - 1) < 1e-12
assert all(weight == 0 for weight in weights[position + 1:])
assert causal_weights(changed, 0) == [1.0, 0, 0, 0, 0, 0]
print("因果 mask 和归一化检查通过")


## 验收与答案
后面三个位置都是 0；提高“猫”的分数后，“小”和“坐”的权重都下降。位置 0 只能看到自己，所以不受“猫”的分数影响。试把 position 改成 5，重新从第一格运行。再读第 06 课里的 QK 点积、mask、softmax 和 V 矩阵乘法；本页直接给定分数，没有实现 Q/K 投影或多头。